# 🏠 Ejercicio Integrador — Regresión: Precio de Propiedades

**Curso:** Taller de programación  
**Temas:** Modelado · Sesgo y Varianza · Explicabilidad · Equidad

---

## Contexto

Una empresa de tasación inmobiliaria quiere construir un modelo para predecir el precio de venta de propiedades. El dataset incluye características físicas del inmueble **y** variables del barrio.

## Variables del dataset

| Variable | Descripción |
|----------|-------------|
| `m2` | Superficie en m² |
| `cuartos` | Dormitorios |
| `banos` | Baños |
| `garage` | 1 si tiene garage, 0 si no |
| `antiguedad` | Años de antigüedad |
| `pisos` | Cantidad de pisos del edificio |
| `piso` | Piso en el que está ubicado el departamento |
| `ingreso_barrio` | Ingreso promedio del barrio (miles $) |
| `dist_centro` | Distancia al centro en km |
| `zona` | **Variable sensible:** A (alto valor), B (medio), C (bajo valor) |
| `precio` | **Target:** precio de venta en miles $ |

---
## Parte 0 — Setup y Dataset

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import warnings
warnings.filterwarnings('ignore')
matplotlib.rcParams['figure.dpi'] = 110

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

np.random.seed(42)

In [ ]:
def generar_dataset(n=600, seed=42):
    rng = np.random.default_rng(seed)
    n_a, n_b, n_c = n//3, n//3, n - 2*(n//3)

    def bloque(n_z, zona_label, m2_mu, ingreso_mu, dist_mu, premio):
        m2         = rng.normal(m2_mu, 20, n_z).clip(30, 300)
        cuartos    = rng.integers(1, 6, n_z).astype(float)
        banos      = (cuartos * rng.uniform(0.4, 0.8, n_z)).round().clip(1, 4)
        garage     = rng.binomial(1, 0.5 + 0.2*(zona_label=='A') - 0.1*(zona_label=='C'), n_z).astype(float)
        antiguedad = rng.uniform(0, 40, n_z).round()
        pisos      = rng.integers(1, 15, n_z).astype(float)
        piso       = np.array([rng.integers(0, int(p)+1) for p in pisos], dtype=float)
        ingreso_b  = rng.normal(ingreso_mu, 15, n_z).clip(20, 300)
        dist_c     = rng.normal(dist_mu, 2, n_z).clip(0.5, 30)
        zona       = np.array([zona_label] * n_z)
        precio = (
            800*m2 + 18000*cuartos + 12000*banos + 25000*garage
            - 800*antiguedad + 500*pisos + 1500*piso + 300*ingreso_b
            - 2000*dist_c + premio + rng.normal(0, 15000, n_z)
        ) / 1000
        return pd.DataFrame({
            'm2': m2, 'cuartos': cuartos, 'banos': banos,
            'garage': garage, 'antiguedad': antiguedad, 'pisos': pisos,
            'piso': piso, 'ingreso_barrio': ingreso_b, 'dist_centro': dist_c,
            'zona': zona, 'precio': precio.clip(50)
        })

    df = pd.concat([
        bloque(n_a, 'A', m2_mu=110, ingreso_mu=180, dist_mu=3,  premio= 80000),
        bloque(n_b, 'B', m2_mu= 85, ingreso_mu=100, dist_mu=8,  premio=     0),
        bloque(n_c, 'C', m2_mu= 65, ingreso_mu= 50, dist_mu=15, premio=-60000),
    ], ignore_index=True).sample(frac=1, random_state=seed).reset_index(drop=True)
    return df

df = generar_dataset()
print(f"Dataset: {df.shape}")
print(df.groupby('zona')['precio'].agg(['count','mean','std']).round(1))

### 📋 Consigna 0.1 — EDA

a. ¿Cuál es la distribución del precio? ¿Es simétrica?  
b. ¿Qué variables tienen mayor correlación con el precio?  
c. ¿Difieren los precios y el m² promedio entre zonas?

**✏️ Interpretación:**

---
## Parte 1 — Modelado

In [ ]:
features = ['m2', 'cuartos', 'banos', 'garage', 'antiguedad',
            'pisos', 'piso', 'ingreso_barrio', 'dist_centro']

df_enc = pd.get_dummies(df, columns=['zona'], drop_first=False)
features_con_zona = features + ['zona_A', 'zona_B', 'zona_C']

X = df_enc[features_con_zona]
y = df['precio']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Train: {X_train.shape}   Test: {X_test.shape}")

### 📋 Consigna 1.1 — Entrenar y comparar modelos

Entrenar los tres modelos y completar la tabla:

| Modelo | MAE train | MAE test | RMSE test | R² test | Gap MAE |
|--------|-----------|----------|-----------|---------|--------|
| Regresión Lineal | | | | | |
| Árbol (depth=5) | | | | | |
| Random Forest | | | | | |

- ¿Qué modelo tiene mayor gap entre train y test? ¿Qué indica?
- ¿Cuál elegirías para producción?

**✏️ Interpretación:**

### 📋 Consigna 2.1 — Estabilidad del modelo

Entrenar árbol (depth=5) y Random Forest 10 veces con distintas semillas. Calcular R² en cada caso.

- ¿Cuál modelo es más estable?
- ¿Cómo se relaciona con el concepto de varianza del estimador?

**✏️ Interpretación:**

---
## Parte 3 — Explicabilidad con SHAP

### 📋 Consigna 3.1 — Importancia global

Graficar el bar plot de importancia SHAP y compararlo con la importancia clásica del RF.  
¿Las variables de zona aparecen entre las más importantes?

**✏️ Interpretación:**

### 📋 Consigna 3.2 — Beeswarm

¿Cómo afecta `m2` al precio? ¿Y `antiguedad`?  
¿Qué te dice el beeswarm que el bar plot no puede?

**✏️ Interpretación:**

### 📋 Consigna 3.3 — Explicación individual

Mostrar el waterfall para la propiedad con predicción más alta y la más baja del test set.  
¿Qué feature es más determinante en cada caso?

**✏️ Interpretación:**

---
## Parte 4 — Sesgo, Varianza y Equidad

### 📋 Consigna 4.1 — Sesgo y varianza

Usando la librería `mlxtend`, calcular la descomposición bias-varianza del MSE para el árbol (depth=5) y el Random Forest.

$$\text{MSE} \approx \text{Bias}^2 + \text{Varianza}$$

- ¿Qué componente reduce principalmente el RF respecto al árbol?
- ¿Es consistente con lo observado en la consigna 2.1?

**✏️ Interpretación:**

### 📋 Consigna 4.2 — Equidad

Usando la librería `dalex`, analizar la equidad del Random Forest tomando `zona` como variable sensible y **Zona C como grupo de referencia**.

- ¿Qué métricas detecta como sesgadas?
- ¿Qué zona presenta mayor error relativo respecto a Zona C?
- ¿Por qué tiene sentido usar Zona C como referencia en este problema?

**✏️ Interpretación:**

---
## Parte 5 — Síntesis

### 📋 Consigna 5 — Informe ejecutivo

Redactar un informe breve (máximo 250 palabras) respondiendo:

1. ¿Qué modelo recomendás y por qué?
2. ¿Qué variables explican la mayor parte del precio?
3. ¿Presenta el modelo algún problema de equidad?
4. ¿Qué datos adicionales pedirías para mejorar el modelo?

---

**✏️ Informe:**

---